In [1]:
from qiskit.transpiler import CouplingMap, Layout
from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes import SabreLayout, SetLayout, SabreSwap, RemoveFinalMeasurements
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.transpiler.passes import VF2Layout

Select circuit

In [2]:
filename = "circuits/swap_test_n83.qasm"
qc = QuantumCircuit.from_qasm_file(filename)
qc_dag = circuit_to_dag(qc)
qc_dag = RemoveFinalMeasurements().run(qc_dag)
qc = dag_to_circuit(qc_dag)
#qc.draw('mpl', fold=-1)

Select coupling map

In [3]:
from qiskit_ibm_runtime.fake_provider.backends.sherbrooke import FakeSherbrooke
fake_eagle = FakeSherbrooke()
coupling = fake_eagle.coupling_map
coupling_map = CouplingMap(couplinglist=coupling)
coupling_map.make_symmetric()

random_seed = 1

Pass manager setup

In [4]:
basis_gates=["rz", "sx", "x", "cx", "swap"]
pm = generate_preset_pass_manager(optimization_level=3, coupling_map=coupling_map, seed_transpiler=random_seed, basis_gates=basis_gates)
#vf2_layout = VF2Layout(coupling_map=coupling_map, seed=1)
#pm.layout.replace(1, vf2_layout)


Transpile

In [5]:
qc_tr = pm.run(qc)

depth_2q = qc_tr.depth(lambda x: x.operation.num_qubits == 2)
ops = qc_tr.count_ops()

print(f"Transpiled circuit: {filename}")
print(f"  2q depth: {depth_2q}")
print(f"  Ops: {ops}")

Transpiled circuit: circuits/swap_test_n83.qasm
  2q depth: 319
  Ops: OrderedDict({'rz': 625, 'sx': 330, 'cx': 287, 'swap': 153})


In [6]:
coupling_map_all_to_all = CouplingMap().from_full(coupling_map.size())
coupling_map_all_to_all.make_symmetric()



In [7]:
basis_gates=["rz", "sx", "x", "cx", "swap"]
pm_all_to_all  = generate_preset_pass_manager(optimization_level=3, coupling_map=coupling_map_all_to_all, seed_transpiler=random_seed, basis_gates=basis_gates)
#vf2_layout = VF2Layout(coupling_map=coupling_map, seed=1, max_trials=00)
#pm.layout.replace(1, vf2_layout)



In [8]:
qc_tr = pm_all_to_all.run(qc)

depth_2q = qc_tr.depth(lambda x: x.operation.num_qubits == 2)
ops = qc_tr.count_ops()

print(f"Transpiled circuit: {filename}")
print(f"  2q depth: {depth_2q}")
print(f"  Ops: {ops}")

Transpiled circuit: circuits/swap_test_n83.qasm
  2q depth: 207
  Ops: OrderedDict({'rz': 618, 'sx': 330, 'cx': 287})
